## Spark initialization

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkSQL-Kafka-Postgres") \
    .config("spark.jars", "/path/to/postgresql-42.7.1.jar") \
    .getOrCreate()

## Read from PostgreSQL

In [ ]:
df_pg = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "(SELECT \"CompraManha\", \"VendaManha\", \"PUCompraManha\", \"PUVendaManha\", \"PUBaseManha\", \"Data_Vencimento\", \"Data_Base\", \"Tipo\", dt_update FROM public.dadostesouroipca) AS dados") \
    .option("user", "postgres") \
    .option("password", "postgres") \
    .load()

df_pg.show()
df_pg.createOrReplaceTempView("tabela_ipca")

## SQL query over the PostgreSQL data

In [ ]:
spark.sql("""
    SELECT Tipo, COUNT(*) AS total
    FROM tabela_ipca
    GROUP BY Tipo
""").show()

## Read from Kafka

In [ ]:
df_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "topic_name") \
    .load()

df_kafka_str = df_kafka.selectExpr("CAST(value AS STRING)")
df_kafka_str.show()
df_kafka_str.createOrReplaceTempView("mensagens_kafka")

## SQL query over the Kafka data

In [ ]:
spark.sql("""
    SELECT value, LENGTH(value) as tamanho
    FROM mensagens_kafka
""").show()

## Shutdown

In [ ]:
spark.stop()